<a href="https://colab.research.google.com/github/aiportfoliorhea/ai-portfolio/blob/main/loanbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn chromadb langchain pypdf sentence-transformers langchain-community langchain-text-splitters langchain-chroma anthropic

In [ ]:
import chromadb
import langchain
import fastapi

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_12445/3590037574.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="loanbot",
    embedding_function=embeddings,
)

print("Vector store ready")

Vector store ready


In [13]:
# 2. Create a simple test file to work with

sample_text = """
LOAN AGREEMENT

Section 1: Loan Terms
The borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.

Section 2: Prepayment
Borrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.

Section 3: Late Payment
Payments received more than 10 days after the due date will incur a late fee of $150 or 3% of the payment amount, whichever is greater.

Section 4: Default
Borrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.
"""

# Save it as a text file
with open("sample_loan.txt", "w") as f:
    f.write(sample_text)

print("Sample loan doc created")

Sample loan doc created ✓


In [15]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("sample_loan.txt")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")


Loaded 1 document(s)


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=10)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")

Split into 5 chunks


In [18]:
for i in range(len(chunks)):
    print(f"Chunk {i}: {chunks[i]}")

Chunk 0: page_content='LOAN AGREEMENT' metadata={'source': 'sample_loan.txt'}
Chunk 1: page_content='Section 1: Loan Terms
The borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.' metadata={'source': 'sample_loan.txt'}
Chunk 2: page_content='Section 2: Prepayment
Borrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.' metadata={'source': 'sample_loan.txt'}
Chunk 3: page_content='Section 3: Late Payment
Payments received more than 10 days after the due date will incur a late fee of $150 or 3% of the payment amount, whichever is greater.' metadata={'source': 'sample_loan.txt'}
Chunk 4: page_content='Section 4: Default
Borrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.' metadata={'source': 'sample_loan.tx

In [19]:
vector_store.add_documents(chunks)
print("Chunks stored in vector store")

Chunks stored in vector store


In [22]:
retriever = vector_store.as_retriever()
results = retriever.invoke("what is the loan amount?")
print(results)

[Document(id='1d50b9ad-eb33-4231-b849-f299fc331f75', metadata={'source': 'sample_loan.txt'}, page_content='Section 1: Loan Terms\nThe borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.'), Document(id='09862c62-f5a1-49f0-bcdf-04aa84838dcf', metadata={'source': 'sample_loan.txt'}, page_content='LOAN AGREEMENT'), Document(id='de0a29e6-5f75-42e3-9ed1-1a5a84fe43f9', metadata={'source': 'sample_loan.txt'}, page_content='Section 4: Default\nBorrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.'), Document(id='6e6f7805-ee92-4b94-ae0f-f3ad147b22ac', metadata={'source': 'sample_loan.txt'}, page_content='Section 2: Prepayment\nBorrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.')]


In [23]:
for doc in results:
    print(doc.page_content)
    print("---")

Section 1: Loan Terms
The borrower agrees to repay a principal amount of $500,000 at a fixed interest rate of 7.2% per annum over a period of 15 years. Monthly payments shall be $4,561.
---
LOAN AGREEMENT
---
Section 4: Default
Borrower is considered in default after 60 days of non-payment. Upon default, the full remaining balance becomes immediately due.
---
Section 2: Prepayment
Borrower may prepay any amount without penalty after the first 12 months. Prepayment within the first 12 months incurs a fee of 2% of the remaining balance.
---


In [30]:
import anthropic
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"]  = userdata.get('anthropic_key')
print("fetched Anthropic key succesfully")

fetched Anthropic key succesfully


In [34]:
from anthropic import Anthropic

client = Anthropic()

def ask_loanbot(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""Act as a legal loan document assistant. Answer the question using only the context provided.
If the answer is not in the context, say "I don't have that information in the document. Don't make up any anwers, strictly stick to the context given to you."

Context:
{context}

Question: {question}"""
            }
        ]
    )

    return message.content[0].text

print(ask_loanbot("what is the loan amount?"))
print("----------- ----------- -----------")
print(ask_loanbot("what is the weather in Dallas?"))


Based on the context provided, the loan amount is **$500,000**. 

This is stated in Section 1: Loan Terms, which specifies "The borrower agrees to repay a principal amount of $500,000."
----------- ----------- -----------
I don't have that information in the document. The document provided is a loan agreement and does not contain any information about weather conditions in Dallas or any other location.
